# Лабораторная 2 Часть 1 Конвейер Pandas

**Иноземцев Сергей Викторович · ЕТ-216 · вариант 11**

Источник задания: «Лб2 (ч1).pdf». Все вычисления воспроизводимы при последовательном запуске ячеек. Фиксированные random seed заданы в коде.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'datasets'/'StudentsPerformance.csv').is_file())
DATA = BASE/'datasets'
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
def show(name):
    plt.tight_layout()
    plt.savefig(FIG/(name+'.png'), bbox_inches='tight')
    plt.show()

## Формирование и сохранение данных
Создать 15 заказов из условия и сохранить CSV без индекса.

In [ ]:
raw_data={'order_id':list(range(1001,1016)),
'channel':['site','app','app','marketplace','site','app','site','marketplace','app','site','marketplace','app','site','site','marketplace'],
'order_sum':[1200,2500,1800,3200,900,1500,2800,45000,2100,1700,3600,1100,2400,1900,3300],
'delivery_time':[25,np.nan,40,55,20,30,np.nan,70,35,45,50,18,27,np.nan,60],
'rating':[5,4,5,3,5,4,2,1,5,4,3,5,4,5,2]}
df=pd.DataFrame(raw_data);df.to_csv('online_store_orders.csv',index=False);display(df)

index=False исключает технический индекс строк из CSV. Без него при обратной загрузке появился бы дополнительный столбец Unnamed: 0.

## Загрузка и первичная проверка

In [ ]:
store_df=pd.read_csv('online_store_orders.csv')
display(store_df.head());display(store_df.tail());print(store_df.shape,store_df.columns.tolist());store_df.info()
display(store_df.isna().sum());display(store_df.describe())
assert store_df.shape==(15,5)

15 строк, 5 столбцов. Числовые: order_id, order_sum, delivery_time, rating. Три пропуска находятся только в delivery_time; channel — категориальный признак. Максимальная сумма 45000 заметно превосходит остальные заказы.

## Выборка и фильтрация

In [ ]:
display(store_df[['order_sum','delivery_time']])
display(store_df.iloc[3:8,store_df.columns.get_indexer(['order_sum','delivery_time'])])
premium_orders=store_df[(store_df.order_sum>2000)&(store_df.rating==5)];display(premium_orders)

iloc[3:8] включает индексы от 3 до 7. Оператор & объединяет булевы Series поэлементно; and требует единственное логическое значение и для Series неоднозначен. Под фильтр попадает заказ 1009.

## Статистики и группировка

In [ ]:
mean_delivery=store_df.delivery_time.mean();median_delivery=store_df.delivery_time.median()
print('Доставка среднее/медиана:',mean_delivery,median_delivery)
display(store_df.rating.value_counts())
store_df['is_high_value']=store_df.order_sum>3000
display(store_df[['order_id','order_sum','is_high_value']])
channel_stats=store_df.groupby('channel').order_sum.mean().sort_values(ascending=False);display(channel_stats)
print('Высокая стоимость:',store_df.is_high_value.sum())
print('Средняя сумма с выбросом/без:',store_df.order_sum.mean(),store_df.loc[store_df.order_sum<45000,'order_sum'].mean())

В наборе 15 заказов и 3 пропуска времени доставки; среднее время 39.58 мин, медиана 37.50 мин (пропуски исключены). Средняя сумма по каналам: marketplace: 13775.00 руб.; site: 1816.67 руб.; app: 1800.00 руб. Четыре заказа дороже 3000 руб. Заказ на 45000 руб. сильно повышает среднее marketplace; это не доказывает преимущество типичного заказа этого канала. Чаще всего встречается оценка 5. Среднее чувствительнее медианы к экстремальным значениям.

## Дополнительное задание Гистограмма

In [ ]:
store_df.order_sum.plot.hist(bins=15);plt.title('Сумма заказа с крупным наблюдением');plt.xlabel('Рубли');plt.ylabel('Число заказов');show('order_sum')

Заказ 45000 растягивает ось и сжимает основную массу распределения. Для дополнительной детализации полезны логарифмическая ось или отдельный график основной группы, но исходное наблюдение нельзя удалять без проверки.